In [ ]:
# 02_cleaning_eda.ipynb — Cell A: inspect raw CSVs before schema design
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
RAW = PROJECT_ROOT / "data" / "raw"

files = [
    "entsoe_generation_AT.csv",
    "entsoe_load_AT.csv",
    "entsoe_prices_AT.csv",
    "weather_vienna.csv",
]

for fname in files:
    path = RAW / fname
    if not path.exists():
        print(f"!! MISSING: {path}\n")
        continue
    size_kb = path.stat().st_size / 1024
    print("=" * 80)
    print(f"  {fname}   ({size_kb:,.0f} KB)")
    print("=" * 80)
    with open(path) as f:
        for i, line in enumerate(f):
            if i >= 5:
                break
            print(f"L{i}: {line.rstrip()[:160]}")
    print()

In [ ]:
# 02_cleaning_eda.ipynb — Cell B: open DuckDB + create schema
import duckdb
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
PROCESSED = PROJECT_ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

DB_PATH = PROCESSED / "austria_energy.duckdb"
con = duckdb.connect(str(DB_PATH))

# Clean slate — drop everything for safe re-runs while iterating
for tbl in ["generation_15min", "demand_15min",
            "generation", "demand", "prices", "weather", "owid_energy_at"]:
    con.execute(f"DROP TABLE IF EXISTS {tbl}")

## This version is just for DuckDB/PostgreSQL dialect
## For example, for generation_15min, the following is the correct syntax:

## con.execute("""
## CREATE OR REPLACE TABLE generation_15min (
##     ts_utc      TIMESTAMPTZ NOT NULL,
##     fuel_type   VARCHAR     NOT NULL,
##     flow        VARCHAR     NOT NULL,
##     mw          DOUBLE,
##     PRIMARY KEY (ts_utc, fuel_type, flow)
## )
## """)

# ── Staging (15-min, long format) ────────────────────────────────────────
con.execute("""
CREATE TABLE generation_15min (
    ts_utc      TIMESTAMPTZ NOT NULL,
    fuel_type   VARCHAR     NOT NULL,
    flow        VARCHAR     NOT NULL,      -- 'generation' or 'consumption'
    mw          DOUBLE,
    PRIMARY KEY (ts_utc, fuel_type, flow)
)
""")

con.execute("""
CREATE TABLE demand_15min (
    ts_utc     TIMESTAMPTZ PRIMARY KEY,
    demand_mw  DOUBLE          -- allow NULL; we'll count gaps in QA
)
""")

# ── Final hourly tables (no resampling needed — native hourly) ──────────
con.execute("""
CREATE TABLE prices (
    ts_utc         TIMESTAMPTZ PRIMARY KEY,
    price_eur_mwh  DOUBLE       -- allow NULL; we'll count gaps in QA
)
""")

con.execute("""
CREATE TABLE weather (
    ts_utc          TIMESTAMPTZ PRIMARY KEY,
    temperature_c   DOUBLE,
    solar_wm2       DOUBLE,
    wind_kmh        DOUBLE,
    precip_mm       DOUBLE,
    cloudcover_pct  DOUBLE
)
""")

# generation, demand, owid_energy_at are created via CTAS in later cells.

print(con.sql("SHOW TABLES").df())

In [ ]:
# 02_cleaning_eda.ipynb — Cell C: load OWID + prices + weather
import pandas as pd

RAW = PROJECT_ROOT / "data" / "raw"

# ── 1. OWID — annual, no TZ, CTAS straight from CSV ─────────────────────
con.execute(f"""
CREATE TABLE owid_energy_at AS
SELECT *
FROM read_csv_auto('{RAW / "owid_energy_AUT.csv"}', header=true)
""")
n = con.sql("SELECT COUNT(*) FROM owid_energy_at").fetchone()[0]  # pyright: ignore[reportOptionalSubscript]
print(f"owid_energy_at: {n} rows")

# ── 2. Prices — timestamps already carry offsets, parse to UTC ──────────
df_p = pd.read_csv(RAW / "entsoe_prices_AT.csv", index_col=0, parse_dates=True)
df_p.index = pd.to_datetime(df_p.index, utc=True)         # to UTC
df_p = df_p.rename_axis("ts_utc").reset_index()
df_p.columns = ["ts_utc", "price_eur_mwh"]
# there exists an explicit form of insert
# It is for assuring to be name sensitive about columns and not just order
con.execute("INSERT INTO prices SELECT * FROM df_p")
n = con.sql("SELECT COUNT(*) FROM prices").fetchone()[0]  # pyright: ignore[reportOptionalSubscript]
print(f"prices: {n} rows")

# ── 3. Weather — already UTC, just tag the index ────────────────────────
df_w = pd.read_csv(RAW / "weather_vienna.csv",
                   index_col="timestamp", parse_dates=True)
df_w.index = df_w.index.tz_localize("UTC")     # naive → UTC label, no shift  # pyright: ignore[reportAttributeAccessIssue]
df_w = df_w.rename(columns={
    "temperature_2m":      "temperature_c",
    "shortwave_radiation": "solar_wm2",
    "windspeed_10m":       "wind_kmh",
    "precipitation":       "precip_mm",
    "cloudcover":          "cloudcover_pct",
}).rename_axis("ts_utc").reset_index()
con.execute("INSERT INTO weather SELECT * FROM df_w")
n = con.sql("SELECT COUNT(*) FROM weather").fetchone()[0]  # pyright: ignore[reportOptionalSubscript]
print(f"weather: {n} rows")

In [ ]:
# 02_cleaning_eda.ipynb — Cell D: sanity check what's in the DB - Part 1
con.sql("""
SELECT 'prices'  AS tbl, COUNT(*) AS rows,
       MIN(ts_utc) AS first_ts, MAX(ts_utc) AS last_ts
FROM prices
UNION ALL
SELECT 'weather', COUNT(*), MIN(ts_utc), MAX(ts_utc) FROM weather
UNION ALL
SELECT 'owid',    COUNT(*), NULL, NULL FROM owid_energy_at
""").df()

In [ ]:
# 02_cleaning_eda.ipynb — Cell D: sanity check what's in the DB - Part 2
con.sql("""
SELECT 'demand_15min' AS tbl,
       COUNT(*)                              AS rows,
       COUNT(*) FILTER (WHERE demand_mw IS NULL) AS nulls,
       ROUND(100.0 * COUNT(*) FILTER (WHERE demand_mw IS NULL) / COUNT(*), 3)
                                             AS pct_null
FROM demand_15min
UNION ALL
SELECT 'generation_15min',
       COUNT(*),
       COUNT(*) FILTER (WHERE mw IS NULL),
       ROUND(100.0 * COUNT(*) FILTER (WHERE mw IS NULL) / COUNT(*), 3)
FROM generation_15min
""").df()

In [ ]:
# 02_cleaning_eda.ipynb — Cell E: melt generation wide → long, load demand

# ── 4. Generation — wide MultiIndex CSV → long staging ─────────────────────
df_g = pd.read_csv(
    RAW / "entsoe_generation_AT.csv",
    header=[0, 1],          # 2-row header → MultiIndex columns
    index_col=0,
)
# Mixed offsets across DST → utc=True normalises both to UTC
df_g.index = pd.to_datetime(df_g.index, utc=True)
df_g.index.name = "ts_utc"

# Name the column levels so stack() produces nice column names
df_g.columns.names = ["fuel_type", "_flow_raw"]

# Wide → long: every (fuel_type, flow) column becomes its own row
df_g_long = (
    df_g
    .stack(level=["fuel_type", "_flow_raw"], future_stack=True)
    .rename("mw")  # pyright: ignore[reportArgumentType]
    .reset_index()
    .dropna(subset=["mw"])      # drop rows ENTSO-E didn't report
)

# Map ENTSO-E flow labels → our schema
flow_map = {
    "Actual Aggregated":  "generation",
    "Actual Consumption": "consumption",
}
df_g_long["flow"] = df_g_long["_flow_raw"].map(flow_map)

# Defensive: error if ENTSO-E ships a new flow label we haven't handled
unmapped = df_g_long["flow"].isna()
if unmapped.any():
    raise ValueError(
        f"Unknown ENTSO-E flow types: "
        f"{df_g_long.loc[unmapped, '_flow_raw'].unique().tolist()}"
    )

df_g_long = df_g_long[["ts_utc", "fuel_type", "flow", "mw"]]

con.execute("""
    INSERT INTO generation_15min (ts_utc, fuel_type, flow, mw)
    SELECT ts_utc, fuel_type, flow, mw FROM df_g_long
""")
n = con.sql("SELECT COUNT(*) FROM generation_15min").fetchone()[0]  # pyright: ignore[reportOptionalSubscript]
print(f"generation_15min: {n:,} rows")


# ── 5. Demand — single-column CSV, straight-through ────────────────────────
df_d = pd.read_csv(
    RAW / "entsoe_load_AT.csv",
    index_col=0,
)
df_d.index = pd.to_datetime(df_d.index, utc=True)
df_d = df_d.rename_axis("ts_utc").reset_index()
df_d.columns = ["ts_utc", "demand_mw"]

con.execute("""
    INSERT INTO demand_15min (ts_utc, demand_mw)
    SELECT ts_utc, demand_mw FROM df_d
""")
n = con.sql("SELECT COUNT(*) FROM demand_15min").fetchone()[0]  # pyright: ignore[reportOptionalSubscript]
print(f"demand_15min: {n:,} rows")

In [ ]:
# 02_cleaning_eda.ipynb — Cell F: re-run QA on the populated tables
con.sql("""
SELECT 'generation_15min' AS tbl,
       COUNT(*)                                  AS rows,
       COUNT(DISTINCT fuel_type)                 AS n_fuels,
       COUNT(*) FILTER (WHERE mw IS NULL)        AS nulls,
       MIN(ts_utc) AS first_ts, MAX(ts_utc) AS last_ts
FROM generation_15min
UNION ALL
SELECT 'demand_15min', COUNT(*), NULL,
       COUNT(*) FILTER (WHERE demand_mw IS NULL),
       MIN(ts_utc), MAX(ts_utc)
FROM demand_15min
""").df()

In [ ]:
# 02_cleaning_eda.ipynb — Cell G: resample 15-min → hourly via SQL

# ── 6. Generation hourly: filter junk consumption rows + aggregate ─────────
con.execute("""
CREATE OR REPLACE TABLE generation AS
SELECT
    time_bucket(INTERVAL 1 HOUR, ts_utc) AS ts_utc,
    fuel_type,
    flow,
    AVG(mw)   AS mw,            -- MW is instantaneous power → average, don't sum
    COUNT(*)  AS n_quarters     -- expect 4 per row; <4 means source gap
FROM generation_15min
WHERE flow = 'generation'                                       -- always keep
   OR (flow = 'consumption' AND fuel_type = 'Hydro Pumped Storage')
                                                                -- only pumped storage actually consumes
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3
""")
n_g = con.sql("SELECT COUNT(*) FROM generation").fetchone()[0]  # pyright: ignore[reportOptionalSubscript]
print(f"generation: {n_g:,} rows")

# ── 7. Demand hourly ────────────────────────────────────────────────────────
con.execute("""
CREATE OR REPLACE TABLE demand AS
SELECT
    time_bucket(INTERVAL 1 HOUR, ts_utc) AS ts_utc,
    AVG(demand_mw)  AS demand_mw,
    COUNT(*)        AS n_quarters
FROM demand_15min
GROUP BY 1
ORDER BY 1
""")
n_d = con.sql("SELECT COUNT(*) FROM demand").fetchone()[0]  # pyright: ignore[reportOptionalSubscript]
print(f"demand: {n_d:,} rows")

In [ ]:
# Quarter-count check: should always be 4
display(con.sql("""
SELECT 'generation' AS tbl, n_quarters, COUNT(*) AS n_rows
FROM generation
GROUP BY n_quarters
UNION ALL
SELECT 'demand', n_quarters, COUNT(*)
FROM demand
GROUP BY n_quarters
ORDER BY tbl, n_quarters
""").df())

# What's actually in Austria's generation mix?
display(con.sql("""
SELECT
    fuel_type,
    flow,
    ROUND(AVG(mw))      AS avg_mw,
    ROUND(MAX(mw))      AS peak_mw,
    ROUND(SUM(mw) / 1e6, 2) AS total_twh   -- MW × hours / 1e6 = TWh
FROM generation
GROUP BY fuel_type, flow
ORDER BY avg_mw DESC NULLS LAST
""").df())

In [ ]:
con.sql("""
SELECT fuel_type,
       COUNT(DISTINCT mw)         AS n_unique_values,
       ROUND(STDDEV(mw), 4)       AS std_mw,
       ROUND(MIN(mw), 2)          AS min_mw,
       ROUND(MAX(mw), 2)          AS max_mw
FROM generation
WHERE flow = 'generation'
GROUP BY fuel_type
ORDER BY n_unique_values
""").df()

## Phase 2 complete

DuckDB file: `data/processed/austria_energy.duckdb`

| Table | Grain | Source |
|---|---|---|
| `generation`        | hourly × (fuel_type, flow) | ENTSO-E (resampled from 15-min) |
| `demand`            | hourly                     | ENTSO-E (resampled from 15-min) |
| `prices`            | hourly                     | ENTSO-E day-ahead              |
| `weather`           | hourly                     | Open-Meteo / ERA5 (Vienna)     |
| `owid_energy_at`    | annual                     | Our World in Data              |

All timestamps stored as UTC TIMESTAMPTZ. Staging tables (`generation_15min`,
`demand_15min`) retained for reproducibility. Resampling: `AVG(mw)` over
`time_bucket(INTERVAL 1 HOUR, ts_utc)`.

Notable: hydro dominates Austria's mix; coal phased out April 2020;
solar peak-to-avg ratio ~19× drives the duck curve dynamics.